<div style="background: linear-gradient(135deg, #0a0510 0%, #1a0a2e 35%, #0f2d3a 70%, #0a1525 100%);padding: 60px 40px; border-radius: 18px; text-align: center; margin-bottom: 10px; border: 1px solid #ff6b3544;"><h1 style="color: #ff6b35; font-family: 'Courier New', monospace; font-size: 2.9em;letter-spacing: 5px; margin: 0 0 12px 0; text-shadow: 0 0 40px #ff6b3566;">✏️ OpenCV DEVAM</h1><h2 style="color: #a8dadc; font-family: 'Courier New', monospace;font-size: 1.35em; margin: 0 0 18px 0; font-weight: 300;">10. Hafta — Affine & Perspektif Dönüşümler + Çizim Fonksiyonları</h2><p style="color: #6b7280; font-family: 'Courier New', monospace; font-size: 0.85em; margin: 0;">warpAffine &bull; Border Modes &bull; getPerspectiveTransform &bull; line/rect/circle/ellipse &bull; putText &bull; Türkçe Yazı Tipleri</p></div>

---
# 📐 BÖLÜM 1: Affine Dönüşüm — Matematiksel Temel

## 1.1 Affine Dönüşüm Nedir?

Bir görüntüdeki her noktayı doğrusal bir kurala göre başka bir noktaya eşleyen dönüşümdür. **Paralel çizgiler paralel kalır** — affine dönüşümün tanımlayıcı özelliği budur.

```
Genel form:
  [x']   [a  b  tx]   [x]
  [y'] = [c  d  ty] × [y]
                       [1]

6 parametre → 6 serbestlik derecesi:
  a, b, c, d → döndürme + ölçekleme + kesme (shear)
  tx, ty     → öteleme (translation)

Affine'in YAPABİLDİĞİ:
  ✅ Öteleme (translation)
  ✅ Döndürme (rotation)
  ✅ Ölçekleme (scaling)
  ✅ Kesme / yamultma (shear)
  ✅ Aynalama (reflection)

Affine'in YAPAMADIĞI:
  ❌ Perspektif bozulma (paralel çizgiler birleşemez)
  ❌ Lens distorsiyonu düzeltme (doğrusal değil)
```

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Affine Geometrisinin Tarihi:</b><br><br>"Affine" kelimesi Leonhard Euler tarafından 1748'de tanıtıldı, Latin "affinis" (akraba, bağlı) kelimesinden gelir. Dönüşüm sonrası şekillerin orijinalle "akraba" kalması (paralel çizgilerin paralel kalması) özelliğini ifade eder.<br><br>Projektif geometri ise August Ferdinand Möbius tarafından 1827'de sistematize edildi (Möbius bandı ile aynı kişi!). Affine dönüşümler, projektif dönüşümlerin özel bir alt kümesidir — perspektif dönüşümün "sonsuza giden çizgileri koruma" kısıtlamasını kaldırırsanız projektif dönüşüm elde edersiniz.<br><br><b>Modern bağlantı:</b> Bilgisayarlı görüde affine vs projektif (perspektif) ayrımı 1992'de Faugeras'ın "What can be seen in three dimensions with an uncalibrated stereo rig?" makalesiyle netleşti — bu makale 3D rekonstrüksiyon alanını kurdu.</span></div>

## 1.2 Affine vs Perspektif — Görsel Karşılaştırma

```
AFFINE (2×3 matris, 6 parametre):     PERSPEKTİF (3×3 matris, 8 parametre):

┌─────┐                                ┌──────┐
│     │ →  paralelkenar                │      │ →  yamuk/trapez
│     │                                 \      \
└─────┘                                  └──────┘

En az 3 nokta çifti gerekir            En az 4 nokta çifti gerekir
(3 × 2 = 6 bilinmeyen)                 (4 × 2 = 8 bilinmeyen)
```

Bu hafta her iki dönüşümü de göreceğiz.

---
# 🔄 BÖLÜM 2: Rotasyon Matrisi ile Döndürme

## 2.1 `cv.getRotationMatrix2D` ve `cv.warpAffine`

```python
M = cv.getRotationMatrix2D(merkez, aci, olcek)
sonuc = cv.warpAffine(kaynak, M, (genislik, yukseklik))
```

Bu iki fonksiyon birlikte serbest açılı döndürme ve ölçeklemeyi mümkün kılar. `cv.getRotationMatrix2D` 2×3 boyutlu rotasyon matrisini hesaplar, `cv.warpAffine` bu matrisi kaynak görüntüye uygular.

**Üretilen matrisin içeriği:**

```
[α   β   (1-α)×cx - β×cy]
[-β  α   β×cx + (1-α)×cy]

α = olcek × cos(aci)
β = olcek × sin(aci)
cx, cy = merkez koordinatları
```

In [ ]:
import cv2 as cv

resim = cv.imread("resim/manzara.jpg")

satir, sutun = resim.shape[:2]
print(f"Görüntü: {satir} satır × {sutun} sütun")

# Merkez etrafında −45° döndür, ölçeği 0.5'e indir
M = cv.getRotationMatrix2D((sutun/2, satir/2), -45, 0.5)
print("\nRotasyon matrisi M:")
print(M)

# Çıktı canvas'ı 2× büyük → döndürülmüş görüntü sığsın
sonuc = cv.warpAffine(resim, M, (int(sutun*2), int(satir*2)))

cv.imshow("sonuc", sonuc)
cv.waitKey(0)
cv.destroyAllWindows()

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>warpAffine Nasıl Çalışır? — Ters Eşleme (Inverse Mapping):</b><br><br>Sezgisel olarak düşünürsek: "her kaynak pikseli alıp hedefteki yeni konumuna taşıyalım." Ama bu yanlış sonuç verir — hedefte boş pikseller (delikler) oluşur.<br><br><b>Gerçek algoritma:</b> Hedef görüntünün her pikseli için tersi yapılır:<br>1. Hedef piksel koordinatını al (x', y')<br>2. M'nin tersini uygula: (x, y) = M⁻¹ × (x', y', 1)<br>3. Kaynak görüntüde (x, y) konumundaki pikseli oku<br>4. (x, y) tam sayı olmadığı için interpolasyon yap (bilinear varsayılan)<br>5. Hedef pikseli bu değerle doldur<br><br>Bu "pull" yaklaşımı, her hedef pikselin mutlaka bir değer almasını garanti eder — delik kalmaz. Aynı algoritma OpenGL, DirectX ve tüm modern grafik pipelines'ın temelidir.</span></div>

In [ ]:
import cv2 as cv

resim = cv.imread("resim/sudoku.jpg")
satir, sutun = resim.shape[:2]
print(f"Görüntü: {satir}×{sutun}")

M = cv.getRotationMatrix2D((sutun/2, satir/2), 45, 0.5)

# borderValue: dışarıda kalan bölgeye özel renk ata
sonuc = cv.warpAffine(resim, M, (sutun, satir), borderValue=(100, 0, 200))

cv.imshow("sonuc", sonuc)
cv.waitKey(0)
cv.destroyAllWindows()

---
# 🖼️ BÖLÜM 3: Border Mode'lar — Dönüşüm Sonrası Boşlukları Doldurma

## 3.1 Border Mode Seçenekleri

Döndürme/ölçekleme sonrası görüntü çerçevesinin dışında kalan bölgeler oluşur. Bu boş bölgeler farklı yöntemlerle doldurulabilir:

```
  abcdefgh  →  ?????|abcdefgh|?????
  orijinal        ?  ile gösterilen bölgeler nasıl doldurulsun?
```

| Mode | Davranış | Örnek (orjinal=abcdefgh) |
|------|----------|------------------------|
| `BORDER_CONSTANT` | Sabit değer (borderValue ile) | `00000\|abcdefgh\|00000` |
| `BORDER_REPLICATE`| Kenar pikseli tekrar | `aaaaa\|abcdefgh\|hhhhh` |
| `BORDER_REFLECT` | Ayna yansıtma | `fedcb\|abcdefgh\|gfedc` |
| `BORDER_REFLECT_101` | Ayna (kenar hariç) | `gfedc\|abcdefgh\|gfedc` |
| `BORDER_WRAP` | Görüntüyü tekrarla | `cdefg\|abcdefgh\|abcde` |
| `BORDER_TRANSPARENT`| Değiştirme | orijinali koru |

In [ ]:
import cv2 as cv

resim = cv.imread("resim/sudoku.jpg")
satir, sutun = resim.shape[:2]

M = cv.getRotationMatrix2D((sutun/2, satir/2), 45, 0.5)

# BORDER_REFLECT101: kenarları aynalayarak doldurur (en yumuşak geçiş)
sonuc = cv.warpAffine(resim, M, (sutun, satir), borderMode=cv.BORDER_REFLECT101)

cv.imshow("sonuc", sonuc)
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
import cv2 as cv

resim = cv.imread("resim/manzara.jpg")
satir, sutun = resim.shape[:2]

M = cv.getRotationMatrix2D((sutun/2, satir/2), 45, 0.25)

# BORDER_REPLICATE: kenar pikselini sürekli tekrarlar (çizgi izi gibi görünür)
sonuc = cv.warpAffine(resim, M, (int(sutun), int(satir)), borderMode=cv.BORDER_REPLICATE)

cv.imshow("sonuc", sonuc)
cv.waitKey(0)
cv.destroyAllWindows()

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">🔄 Karşılaştırma</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Hangi Border Mode Ne Zaman?</b><br><br><b>BORDER_CONSTANT (siyah kenar):</b> Profesyonel görünüm istemiyorsanız, veya makine öğrenmesi için eğitim verisi hazırlıyorsanız. Model "burada bilgi yok" sinyalini alır.<br><br><b>BORDER_REPLICATE:</b> Yatay panoramalar, belge taraması — kenarlar zaten düz bir alana uzanıyormuş gibi görünür. En yaygın pratik tercih.<br><br><b>BORDER_REFLECT_101:</b> Filtreleme ve konvolüsyon işlemlerinin standardı. Gaussian blur, Sobel vb. OpenCV'nin varsayılanı. Kenar süreksizliğini en aza indirdiği için gürültü üretmez.<br><br><b>BORDER_WRAP:</b> Tekrarlanan doku (texture) oluştururken, gök atlası (skymap) ve çevresel aydınlatma haritalarında.</span></div>

---
# 🔁 BÖLÜM 4: Sürekli Döndürme Animasyonu

## 4.1 Resmi 360° Döndürme — Kırpma Olmadan

`BORDER_REPLICATE` sayesinde kenarlar doldurulur, görüntü hiçbir zaman kırpılmış görünmez:

In [ ]:
import cv2 as cv

resim = cv.imread("resim/manzara.jpg")
satir, sutun = resim.shape[:2]
print(f"Görüntü: {satir}×{sutun}")

print("360° sürekli döndürme başlıyor... (ESC ile çıkın)")

while True:
    for i in range(360, 0, -1):   # 360→1: saat yönünde
        M = cv.getRotationMatrix2D((sutun/2, satir/2), i, 0.5)
        sonuc = cv.warpAffine(resim, M, (sutun, satir), borderMode=cv.BORDER_REPLICATE)

        cv.imshow("sonuc", sonuc)
        if cv.waitKey(10) == 27:   # ESC
            cv.destroyAllWindows()
            quit()

## 4.2 Kameradan Yüz Döndürme Efekti

Kameranın belirli bir bölgesini kırpıp her karede bir derece döndürme:

In [ ]:
import cv2 as cv

video = cv.VideoCapture(0)
i = 0

while video.isOpened():
    ret, frame = video.read()
    if not ret:
        break

    # ROI: yüz bölgesi (kullanıcı kendi kamerasına göre ayarlamalı)
    frame = frame[100:400, 200:700]
    satir, sutun = frame.shape[:2]

    i += 1   # her karede 1 derece daha döndür
    M = cv.getRotationMatrix2D((sutun/2, satir/2), i, 0.5)
    sonuc = cv.warpAffine(frame, M, (sutun, satir), borderMode=cv.BORDER_REPLICATE)

    cv.imshow("sonuc", sonuc)

    if cv.waitKey(10) == ord("q"):
        break

video.release()
cv.destroyAllWindows()

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 Sürekli Döndürme ve İnsan Görsel Sisteminin Sınırları</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">İnsan gözü yaklaşık <b>10-12 Hz</b>'in altındaki hareketleri "kesikli" olarak algılar; üstündekiler sürekli görünür. Bu yüzden filmler 24 FPS, oyunlar 60+ FPS çalışır.<br><br>Yukarıdaki kod <code>waitKey(10)</code> ile yaklaşık 100 FPS hedefler — pratikte OpenCV'nin işlem yükü nedeniyle 40-80 FPS arası bir hızda döner, bu da insan gözü için "pürüzsüz" görünür.<br><br><b>İlginç not:</b> Dönen bir görüntüye uzun süre bakmak beynin hareket algı nöronlarını "doyurur" (motion adaptation). Sonra durduğunuzda sabit görüntü ters yönde dönüyormuş gibi görünür — buna "waterfall illusion" denir. 1834'ten beri bilinir!</span></div>

---
# 🧮 BÖLÜM 5: Genel Affine Dönüşüm — `getAffineTransform`

## 5.1 3 Nokta ile Affine Matrisi Hesaplama

`getRotationMatrix2D` yalnızca döndürme+ölçekleme yapar. Daha genel dönüşümler (shear, skew) için 3 nokta çiftinden matris hesaplanır:

```
kaynak 3 nokta  →  hedef 3 nokta
     (6 denklem)      (6 bilinmeyen)
     a, b, c, d, tx, ty çözülür
```

**Neden tam 3 nokta?** Affine matrisinin 6 parametresi var. Her nokta çifti 2 denklem verir (x ve y). Sistem tam belirlenir: 3 nokta × 2 = 6 denklem = 6 parametre.

In [ ]:
import cv2 as cv
import numpy as np

resim = cv.imread("resim/sudoku.jpg")
satir, sutun = resim.shape[:2]

# Kaynak: orijinal görüntünün 3 referans noktası
kaynak_noktalar = np.float32([
    [0, 0],              # sol üst
    [sutun-1, 0],        # sağ üst
    [0, satir-1]         # sol alt
])

# Hedef: aynı noktalar nereye gitsin? — sağ üstü orta tarafa çekiyoruz
hedef_noktalar = np.float32([
    [0, 0],
    [int(0.5*(sutun-1)), 0],   # sağ üst → genişliğin ortasına
    [int(0.5*(sutun-1)), satir-1]
])

afin_matrisi = cv.getAffineTransform(kaynak_noktalar, hedef_noktalar)
print("Affine matrisi:\n", afin_matrisi)

sonuc = cv.warpAffine(resim, afin_matrisi, (sutun, satir))
cv.imshow("sonuc", sonuc)
cv.waitKey(0)
cv.destroyAllWindows()

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Hangi 3 noktayı seçmeliyim?</b><br><br>Matematiksel olarak herhangi 3 nokta olabilir, ama <b>kollineer olmamaları</b> (aynı doğru üzerinde olmamaları) şarttır. Kollineer 3 nokta doğrusaldır — affine matrisi tek türlü belirlenemez, algoritma başarısız olur.<br><br>Pratikte: <b>Görüntünün 3 farklı köşesini</b> seçin. Bu, maksimum geometrik bilgi sağlar ve sayısal kararlılığı artırır.<br><br>Bilgisayarlı görüde bu nokta çiftlerini manuel değil, genellikle özellik eşleştirme (feature matching — ORB, SIFT) ile otomatik bulunur. Panorama oluşturma algoritmaları tam bu mantıkla çalışır.</span></div>

---
# 🎥 BÖLÜM 6: Perspektif Dönüşüm

## 6.1 `getPerspectiveTransform` ile 4 Noktadan Homografi

Perspektif dönüşüm, affine'in genelleştirmesidir — paralel çizgilerin birleşmesine izin verir (tıpkı uzaktaki tren raylarının ufukta birleşmesi gibi).

```
Affine (2×3):              Perspektif (3×3):
┌───┐                       ┌───┐
│   │                        \   \
└───┘                         └───┘

Paralel çizgiler            Yamuk üretilebilir
paralel kalır                (paralel olmayan yapı)

3 nokta yeterli             4 nokta gerekli
```

```python
M = cv.getPerspectiveTransform(kaynak_4_nokta, hedef_4_nokta)
sonuc = cv.warpPerspective(resim, M, (genislik, yukseklik))
```

In [ ]:
import numpy as np
import cv2

resim = cv2.imread("resim/sudoku.jpg")
satir, sutun, kanal = resim.shape
print(f"Orijinal: {satir}×{sutun}")

# Sudoku'nun 4 köşesi (elle belirlendi)
kaynak_noktalar = np.float32([
    [56, 65],    # sol üst
    [368, 52],   # sağ üst
    [28, 387],   # sol alt
    [389, 390]   # sağ alt
])

# Hedef: mükemmel 300×300 kare
hedef_noktalar = np.float32([
    [0, 0], [300, 0], [0, 300], [300, 300]
])

M = cv2.getPerspectiveTransform(kaynak_noktalar, hedef_noktalar)
print("\nPerspektif (homografi) matrisi (3×3):")
print(M)

sonuc = cv2.warpPerspective(resim, M, (300, 300))

cv2.imshow("", sonuc)
cv2.waitKey(0)
cv2.destroyAllWindows()

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Homografi Matrisinin 8 Serbestlik Derecesi:</b><br><br>Perspektif matrisi 3×3 = 9 eleman içerir ama son eleman (H[2,2]) normalleştirme için 1'e sabitlenir. Geriye 8 bilinmeyen kalır.<br><br>Her nokta çifti 2 denklem verir (x ve y). 4 nokta → 8 denklem → tam belirlenir.<br><br><b>Daha fazla nokta olursa?</b> Sistem aşırı belirlenir (overdetermined). Bu durumda en küçük kareler yöntemi (least squares) veya RANSAC ile en iyi çözüm bulunur. <code>cv.findHomography()</code> fonksiyonu tam bunu yapar ve yanlış eşleşmeleri (outlier) otomatik atar.<br><br><b>Önemli:</b> 4 nokta <b>kollineer olmamalı</b> ve hiçbir 3'ü aynı doğru üzerinde olmamalı — yoksa sistem tekil (singular) olur.</span></div>

## 6.2 Düzeltme Sonrası İnce Hata Giderme — Yamukluk Düzeltme

Perspektif düzeltme sonrası hâlâ küçük yamukluklar kalabilir. Görüntüyü iki parçaya bölüp her birini ayrı düzelterek ince ayar yapabiliriz:

In [ ]:
"""
Önceki örneğin devamı: sudoku resminin üst yamukluğu da düzeltilecek.
"""
import numpy as np
import cv2

resim = cv2.imread("resim/sudoku.jpg")
satir, sutun, kanal = resim.shape

# İlk düzeltme (bölüm 6.1 ile aynı)
kaynak_noktalar = np.float32([[56, 65], [368, 52], [28, 387], [389, 390]])
hedef_noktalar  = np.float32([[0, 0], [300, 0], [0, 300], [300, 300]])

M = cv2.getPerspectiveTransform(kaynak_noktalar, hedef_noktalar)
sonuc = cv2.warpPerspective(resim, M, (300, 300))
cv2.imwrite("resim/sudoku_perspektif_duzeltilmis.jpg", sonuc)

# ── Sol taraf ince düzeltme ────────────────────────────────────────
sol_taraf = sonuc[:, 0:201]
kaynak_noktalar_sol = np.float32([[0, 0], [200, 8], [0, 300], [200, 300]])
hedef_noktalar_sol  = np.float32([[0, 0], [200, 0], [0, 300], [200, 300]])
M_sol = cv2.getPerspectiveTransform(kaynak_noktalar_sol, hedef_noktalar_sol)
sonuc_sol = cv2.warpPerspective(sol_taraf, M_sol, (200, 300))

# ── Sağ taraf ince düzeltme ────────────────────────────────────────
sag_taraf = sonuc[:, 201:]
kaynak_noktalar_sag = np.float32([[0, 8], [100, 0], [0, 300], [100, 300]])
hedef_noktalar_sag  = np.float32([[0, 0], [100, 0], [0, 300], [100, 300]])
M_sag = cv2.getPerspectiveTransform(kaynak_noktalar_sag, hedef_noktalar_sag)
sonuc_sag = cv2.warpPerspective(sag_taraf, M_sag, (100, 300))

# Sol + sağ = bütün
son = cv2.hconcat([sonuc_sol, sonuc_sag])

cv2.imshow("", son)
cv2.waitKey(0)
cv2.destroyAllWindows()

## 6.3 Dosya Taraması — Pratik Uygulama

In [ ]:
import numpy as np
import cv2

resim = cv2.imread("resim/dosya.jpg")
satir, sutun, kanal = resim.shape
print(f"Orijinal: {satir}×{sutun}")

# Belgenin 4 köşesi — fotoğrafa göre manuel belirlendi
kaynak_noktalar = np.float32([
    [435, 124],    # sol üst
    [997, 323],    # sağ üst
    [34, 700],     # sol alt
    [692, 984]     # sağ alt
])
hedef_noktalar = np.float32([
    [0, 0], [600, 0], [0, 600], [600, 600]
])

M = cv2.getPerspectiveTransform(kaynak_noktalar, hedef_noktalar)
sonuc = cv2.warpPerspective(resim, M, (600, 600))

cv2.imshow("", sonuc)
cv2.waitKey(0)
cv2.destroyAllWindows()

<div style="background: #2d0020; border-left: 5px solid #f72585; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f72585; font-size: 1.08em;">🌍 Gerçek Dünya</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Perspektif Dönüşüm — Günlük Uygulamalar:</b><br><br><b>CamScanner, Adobe Scan, Office Lens:</b> Telefonla çekilen belgeleri A4 tarayıcı görüntüsüne dönüştürür. Temel algoritma bu haftaki kodla aynıdır, sadece 4 köşe otomatik tespit edilir.<br><br><b>Banka çeki okuma:</b> Mobil uygulamalar çeki fotoğraflamanıza izin verir, perspektif düzeltme + OCR ile otomatik yatırma yapar.<br><br><b>Sokak görüntülerinden harita:</b> Google Street View fotoğraflarını kuşbakışı harita görünümüne dönüştürme.<br><br><b>Otonom araçlar:</b> "Bird's Eye View" — yol kameralarından sanal kuşbakışı görüntü oluşturma. Şerit takibi bu koordinat sisteminde yapılır.<br><br><b>Sanat ve müze dijitalleştirme:</b> Müzedeki tabloların eğik açılardan çekilmiş fotoğraflarını doğru perspektife getirmek.</span></div>

---
# ✏️ BÖLÜM 7: Görüntü Üzerine Çizim — `cv.line`

## 7.1 Çizgi Parametreleri

```python
cv.line(img, pt1, pt2, color, thickness, lineType)
```

| Parametre | Açıklama |
|-----------|----------|
| `img` | Çizim yapılacak görüntü (yerinde değiştirilir!) |
| `pt1, pt2` | Başlangıç ve bitiş noktaları (x, y) |
| `color` | BGR tuple: `(0, 255, 0)` = yeşil |
| `thickness` | Çizgi kalınlığı (piksel). -1 = doldur (çizgide geçersiz) |
| `lineType` | `LINE_4`, `LINE_8` (varsayılan), `LINE_AA` (anti-alias) |

In [ ]:
import cv2 as cv

resim = cv.imread("resim/kirpilmis_manzara.jpg")

# Yalın çizgi: lineType verilmezse LINE_8 (kalın pikseller, merdiven etkisi)
cv.line(resim, (0, 0), (512, 256), (0, 255, 0), 10)

cv.imshow("a", resim)
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
import cv2 as cv

resim = cv.imread("resim/kirpilmis_manzara.jpg")

# lineType=cv.LINE_AA: anti-aliased, pürüzsüz kenarlar
cv.line(resim, (0, 0), (256, 256), (255, 0, 0), 50, lineType=cv.LINE_AA)

cv.imshow("a", resim)
cv.waitKey(0)
cv.destroyAllWindows()

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 LINE_AA — Xiaolin Wu'nun Algoritması</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Anti-aliasing (kenar yumuşatma) 1991'de Xiaolin Wu tarafından geliştirildi.<br><br><b>Sorun:</b> Bilgisayar ekranı piksel matrisi; çizgi tam bir pikselin merkezinden geçmezse "merdiven" efekti (jaggies) oluşur. Bresenham's line algorithm (1965) her pikseli ya tam açık ya tam kapalı yapar — hızlı ama estetik değil.<br><br><b>Wu'nun çözümü:</b> Çizginin bir pikselden ne kadar geçtiğine göre o pikselin parlaklığını ayarlamak. Piksel %30 kaplanmışsa parlaklık %30.<br><br>Bu küçük değişiklik görsel kaliteyi dramatik artırır. Bilgisayar grafiklerinde çığır açan bir fikirdir.<br><br><b>Trade-off:</b> LINE_AA, LINE_8'den yaklaşık 5-10× daha yavaştır. Gerçek zamanlı çizim yapıyorsanız (video oyunları, otonom araçlar) LINE_8 tercih edilir.</span></div>

---
# ▭ BÖLÜM 8: Dikdörtgen Çizme — `cv.rectangle`

```python
cv.rectangle(img, pt1, pt2, color, thickness, lineType)
```

- `pt1`: sol üst köşe
- `pt2`: sağ alt köşe
- `thickness = -1` → **dolu dikdörtgen**
- `thickness > 0` → boş çerçeve, belirtilen kalınlıkta

In [ ]:
import cv2 as cv

resim = cv.imread("resim/kirpilmis_manzara.jpg")

# Sol üst (50, 50), sağ alt (500, 250), BGR (255,100,50) ≈ mavi-turuncu karışımı
cv.rectangle(resim, (50, 50), (500, 250), (255, 100, 50), 10, lineType=cv.LINE_AA)

cv.imshow("a", resim)
cv.waitKey(0)
cv.destroyAllWindows()

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Nesne Tespitinde En Çok Kullanılan Çizim Fonksiyonu!</b><br><br>YOLO, Faster R-CNN, SSD gibi tüm nesne tespit modellerinin çıktısı "bounding box"lardır. Bu kutuları görüntü üzerinde göstermek için <code>cv.rectangle</code> kullanılır:<br><br><code>for kutu in tespit_sonuclari:<br>&nbsp;&nbsp;&nbsp;&nbsp;x1, y1, x2, y2, skor = kutu<br>&nbsp;&nbsp;&nbsp;&nbsp;cv.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)<br>&nbsp;&nbsp;&nbsp;&nbsp;cv.putText(img, f"{skor:.2f}", (x1,y1-5), ...)</code><br><br>Tesla'nın otonom araç ekranları, güvenlik kameraları, Zoom'un yüz takibi — hepsi bu basit fonksiyonla gösterilir.</span></div>

---
# ⭕ BÖLÜM 9: Çember Çizme — `cv.circle`

```python
cv.circle(img, merkez, yaricap, color, thickness, lineType)
```

- Merkez ve yarıçap piksel cinsinden
- `thickness = -1` → **dolu çember (disk)**

In [ ]:
import cv2 as cv

resim = cv.imread("resim/kirpilmis_manzara.jpg")

# 3 çember — merkez konumları farklı, diğer parametreler aynı
cv.circle(resim, (125, 150), 100, (20, 100, 50), 10, lineType=cv.LINE_AA)
cv.circle(resim, (350, 150), 100, (20, 100, 50), 10, lineType=cv.LINE_AA)
cv.circle(resim, (575, 150), 100, (20, 100, 50), 10, lineType=cv.LINE_AA)

cv.imshow("a", resim)
cv.waitKey(0)
cv.destroyAllWindows()

---
# 🥚 BÖLÜM 10: Elips Çizme — `cv.ellipse`

```python
cv.ellipse(img, merkez, (a,b), angle, startAngle, endAngle, color, thickness)
```

| Parametre | Açıklama |
|-----------|----------|
| `(a, b)` | Büyük ve küçük yarı eksen uzunlukları |
| `angle` | Elipsin yönelimi (derece) |
| `startAngle` | Yay başlangıç açısı |
| `endAngle` | Yay bitiş açısı |

`startAngle=0, endAngle=360` → tam elips
Kısmi açılar → **yay** (arc) çizer

In [ ]:
import cv2 as cv

resim = cv.imread("resim/kirpilmis_manzara.jpg")

# (200,200) merkez, a=200, b=50 → yassı elips
# 10° - 350° → 340° yay (sadece 20° boşluk)
cv.ellipse(resim, (200, 200), (200, 50), 0, 10, 350, (255, 0, 0), -1)

cv.imshow("a", resim)
cv.waitKey(0)
cv.destroyAllWindows()

## 10.1 Pacman Animasyonu — Elipsin Yaratıcı Kullanımı

In [ ]:
"""
Elips çizerek pacman animasyonu.
Ağzı açılır kapanır — startAngle ve endAngle değişir.
"""
import cv2 as cv

resim = cv.imread("resim/kirpilmis_manzara.jpg")

i = 10       # ağız açıklığı
artis = 1    # yön: +1 açılıyor, -1 kapanıyor

print("Pacman animasyonu başladı (ESC ile çık)")

while True:
    kaynak = resim.copy()   # orijinali bozma

    # 10+i den 350-i ye arc çiz → ağız etkisi
    cv.ellipse(kaynak, (250, 150), (150, 150), 0, 10+i, 350-i, (255, 0, 0), -1)

    i += artis
    if i > 50 or i < 10:
        artis = -artis   # yönü tersine çevir — salınım!

    cv.imshow("Pacman", kaynak)
    if cv.waitKey(10) == 27:
        break

cv.destroyAllWindows()

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Pacman — 1980'in Sanal Kahramanı:</b><br><br>Pacman, 22 Mayıs 1980'de Japonya'da Namco tarafından piyasaya sürüldü. Tasarımcısı Toru Iwatani, bir pizzanın bir dilimi kesilmiş halinden ilham aldı.<br><br>O dönemde oyunların hemen hepsi savaş temalıydı (Space Invaders, Asteroids). Iwatani, kadınları ve çocukları da çekecek bir oyun istedi — ve başardı. Pacman video oyun tarihinin en tanınır karakterlerinden biri oldu.<br><br>İlginç: Orijinal Pacman'in ağız açılma kapanma animasyonu sadece 4 kare kullanıyordu (hafıza sınırları). Yukarıdaki kod aslında 80+ kareli çok daha yumuşak bir animasyon üretir!</span></div>

---
# ✴️ BÖLÜM 11: Çokgen Çizme — `cv.polylines`

```python
cv.polylines(img, [pts_list], isClosed, color, thickness)
```

- `pts_list`: her eleman bir çokgenin köşe listesi
- `isClosed`: son nokta ile ilk noktayı bağla mı?

**Köşe formatı:** `np.int32` ve shape `(-1, 1, 2)` olmalı

In [ ]:
import cv2 as cv
import numpy as np

resim = cv.imread("resim/kirpilmis_manzara.jpg")

# 3 ayrı doğru parçası — her biri 2 nokta içeriyor
line1 = np.array([[100, 20],  [300, 20]],  np.int32).reshape((-1, 1, 2))
line2 = np.array([[100, 60],  [300, 60]],  np.int32).reshape((-1, 1, 2))
line3 = np.array([[100, 100], [300, 100]], np.int32).reshape((-1, 1, 2))

# Tek polylines çağrısında 3 ayrı çizgiyi çiz
cv.polylines(resim, [line1, line2, line3], True, (0, 255, 255))

cv.imshow("a", resim)
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
import cv2 as cv
import numpy as np

resim = cv.imread("resim/kirpilmis_manzara.jpg")

# 5 köşeli yıldız çizme örneği
kose_sayisi = 5
merkez_x, merkez_y = 300, 150
dis_r = 100
ic_r = 40

koseler = []
for i in range(kose_sayisi * 2):
    aci = i * np.pi / kose_sayisi - np.pi / 2
    r = dis_r if i % 2 == 0 else ic_r
    x = int(merkez_x + r * np.cos(aci))
    y = int(merkez_y + r * np.sin(aci))
    koseler.append([x, y])

yildiz = np.array(koseler, np.int32).reshape((-1, 1, 2))
cv.polylines(resim, [yildiz], isClosed=True, color=(0, 255, 255), thickness=3, lineType=cv.LINE_AA)

# Doldurmak için fillPoly:
# cv.fillPoly(resim, [yildiz], (0, 200, 255))

cv.imshow("yildiz", resim)
cv.waitKey(0)
cv.destroyAllWindows()

---
# 📝 BÖLÜM 12: Metin Ekleme — `cv.putText`

```python
cv.putText(img, text, org, fontFace, fontScale, color, thickness, lineType)
```

| Parametre | Açıklama |
|-----------|----------|
| `text` | Yazılacak metin (sadece ASCII!) |
| `org` | Metnin sol alt köşesi (x, y) |
| `fontFace` | Yazı tipi sabiti |
| `fontScale` | Boyut çarpanı (1.0 = baz) |

### OpenCV'nin Dahili Yazı Tipleri

- `FONT_HERSHEY_SIMPLEX` — standart sans-serif
- `FONT_HERSHEY_PLAIN` — daha küçük/ince
- `FONT_HERSHEY_DUPLEX` — daha kalın
- `FONT_HERSHEY_COMPLEX` — serif tarzı
- `FONT_HERSHEY_TRIPLEX` — kalın serif
- `FONT_HERSHEY_SCRIPT_SIMPLEX` — italik
- `FONT_HERSHEY_SCRIPT_COMPLEX` — kalın italik
- `FONT_ITALIC` | önceki herhangiyle bitwise OR ile italik yap

In [ ]:
import cv2 as cv

resim = cv.imread("resim/kirpilmis_manzara.jpg")

font = cv.FONT_HERSHEY_SIMPLEX

# DİKKAT: Türkçe karakterler kutu/soru işareti görünecek!
cv.putText(resim, "metinÜĞÇÖŞöçışüğ", (10, 200), font, 4, (255, 0, 0), 5,
           lineType=cv.LINE_AA)

cv.imshow("a", resim)
cv.waitKey(0)
cv.destroyAllWindows()

<div style="background: #2d2200; border-left: 5px solid #f9c74f; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f9c74f; font-size: 1.08em;">⚠️ Kritik Nokta</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>cv.putText Türkçe Karakterleri Desteklemez!</b><br><br>OpenCV'nin <code>putText</code> fonksiyonu yalnızca ASCII (127 karakter) destekler. ÜĞÇÖŞıüğ gibi Türkçe karakterler "□" (boş kutu) veya "?" olarak görünür.<br><br><b>Neden?</b> OpenCV'nin Hershey fontları 1960'larda NIST tarafından geliştirildi — yalnızca İngilizce ve matematiksel semboller içerir.<br><br><b>Çözümler:</b><br>1. <b>PIL/Pillow kullan</b> (bölüm 13'te)<br>2. <b>FreeType2 desteği</b> — <code>cv.freetype.createFreeType2()</code> (contrib modülü gerekir)<br>3. <b>İngilizce eşdeğer kullan</b> — U yerine U, Ğ yerine G vb.</span></div>

---
# 🇹🇷 BÖLÜM 13: Türkçe ve Özel Yazı Tipleri — PIL Entegrasyonu

## 13.1 OpenCV + Pillow Hibrit Yaklaşım

Pillow (PIL) tam Unicode desteği sunar — TrueType/OpenType (.ttf, .otf) font dosyalarını doğrudan kullanabilir. Akış:

```
OpenCV (numpy)  →  PIL Image  →  ImageDraw.text()  →  numpy  →  OpenCV
```

In [ ]:
import cv2 as cv
from PIL import ImageFont, Image, ImageDraw
import numpy as np

resim = cv.imread("resim/kirpilmis_manzara.jpg")

# Özel yazı tipi yükle
yazi_tipi_yolu = "yazitipi/Awesome Script Trial.otf"
yazi_tipi = ImageFont.truetype(yazi_tipi_yolu, 60)

# NumPy → PIL Image
img_pil = Image.fromarray(resim)

# Metin çiz (Unicode destekli!)
draw = ImageDraw.Draw(img_pil)
draw.text((50, 80), "Deneme", font=yazi_tipi)
# Türkçe karakter test: draw.text((50, 150), "Çağlayan Güler — İÖĞ", font=yazi_tipi)

# PIL → NumPy
img = np.array(img_pil)
cv.imshow("Türkçe yazı ve farklı yazıtipi kullanımı", img)
cv.waitKey(0)
cv.destroyAllWindows()

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 OpenCV ↔ Pillow — Renk Kanalı Tuzağı</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">OpenCV <b>BGR</b>, Pillow <b>RGB</b> kullanır!<br><br>Bu örnekte <code>Image.fromarray(resim)</code> BGR diziyi RGB olarak yorumlar — ama sonunda <code>np.array(img_pil)</code> ile geri dönünce yine BGR gibi yorumlanır. Bu yüzden renkler görsel olarak doğru görünür.<br><br>Ama <b>PIL'da renk işlemi yapıyorsanız</b> (örn. bir renk filtresi) dikkatli olun:<br><code># Doğru dönüşüm:<br>rgb = cv.cvtColor(bgr, cv.COLOR_BGR2RGB)<br>pil_img = Image.fromarray(rgb)<br># ... PIL işlemleri ...<br>rgb_back = np.array(pil_img)<br>bgr_back = cv.cvtColor(rgb_back, cv.COLOR_RGB2BGR)</code><br><br>Metin çizerken sadece renk <i>değerlerini</i> (yalnızca piksel değerlerini) değiştirdiğimiz için BGR/RGB karışıklığı görünmez — ama bir filtre uygularsanız yanılsama yaşarsınız.</span></div>

---
# 📹 BÖLÜM 14: Canlı Kameraya Dönüşüm Uygulama

## 14.1 Kameradan Sürekli Dönen Görüntü — BORDER_REPLICATE

In [ ]:
"""
1. Affine dönüşümünü kameradan alınan görüntüde dene. Reflect'i de uygula.
2. Kameradan görüntü alarak sürekli döndüren, Replicate uygulayan bir uygulama.
"""
import cv2 as cv

kamera = cv.VideoCapture(0)
ret, resim = kamera.read()
satir, sutun = resim.shape[:2]

print("Kamera döndürme efekti — ESC ile çık")

while True:
    for i in range(360, 0, -1):
        ret, resim = kamera.read()
        if not ret:
            break

        M = cv.getRotationMatrix2D((sutun/2, satir/2), i, 0.5)
        sonuc = cv.warpAffine(resim, M, (sutun, satir), borderMode=cv.BORDER_REPLICATE)

        cv.imshow("sonuc", sonuc)
        if cv.waitKey(33) == 27:
            kamera.release()
            cv.destroyAllWindows()
            quit()

---
# 🏛️ BÖLÜM 15: Üniversite Seviyesi Alıştırmalar

Aşağıdaki sorular bu haftanın konularını ileri düzey bilgisayar görüsü ve mühendislik perspektifiyle birleştirir. **Cevaplar verilmemiştir.**

---
## ❓ Soru 1 — Belge Tarayıcı (CamScanner Benzeri Uygulama)

**Zorluk: ⭐⭐⭐⭐☆**

Kullanıcının kamerayla fotoğrafladığı belgenin dört köşesini **otomatik tespit eden** ve perspektif dönüşüm uygulayarak A4 formatında düzleştirilmiş görüntü üreten bir uygulama yazın.

### Görev:
1. Gri tonlama → Gaussian blur → Canny edge detection ile kenar haritası çıkar
2. `cv.findContours` ile konturları bul
3. `cv.contourArea` ile büyükten küçüğe sırala
4. En büyük konturu `cv.approxPolyDP` ile çokgene yaklaştır
5. Yaklaşım tam **4 köşeli** değilse epsilon değerini dinamik ayarla
6. 4 köşeyi "sol-üst, sağ-üst, sağ-alt, sol-alt" sırasına dizi (x+y toplamı min = sol-üst, max = sağ-alt; x-y: sağ-üst min, sol-alt max)
7. `cv.getPerspectiveTransform + warpPerspective` ile A4 boyutuna getir
8. Son adım: adaptif eşikleme ile "taranmış belge" görünümü oluştur

### Düşünmeniz Gereken Sorular:
- Gerçek fotoğraflarda konturun 4'ten fazla köşeli çıkması durumunda nasıl düzeltirsiniz?
- A4 oranı (1:√2 ≈ 1:1.414) neden özeldir? Kağıdı yarıya katladığınızda neden aynı oranı verir?
- Gölge, kırışık ve el gibi istenmeyen öğeler varken robust tespit nasıl yapılır?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

def koseleri_sirala(pts):
    """4 köşeyi sol-üst, sağ-üst, sağ-alt, sol-alt sırasına diz."""
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]   # sol-üst (x+y en küçük)
    rect[2] = pts[np.argmax(s)]   # sağ-alt
    d = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(d)]   # sağ-üst (y-x en küçük)
    rect[3] = pts[np.argmax(d)]   # sol-alt
    return rect

def belge_tara(resim_yolu):
    # TODO: 8 adımı gerçekle
    pass
```

---
## ❓ Soru 2 — Panorama Oluşturucu — Özellik Tabanlı Stitching

**Zorluk: ⭐⭐⭐⭐⭐**

İki veya daha fazla örtüşen fotoğrafı otomatik olarak birleştirerek panorama görüntüsü oluşturan bir uygulama yazın. `cv.Stitcher` kullanmadan, tüm süreci manuel olarak implement edin.

### Görev:
1. `cv.ORB_create()` ile iki görüntünün özelliklerini tespit et (keypoints + descriptors)
2. `cv.BFMatcher(cv.NORM_HAMMING)` ile eşleştirme yap
3. **Lowe's Ratio Test:** `match.distance < 0.75 × next_match.distance` olanları kabul et
4. `cv.findHomography(srcPts, dstPts, cv.RANSAC)` ile homografi matrisi bul
5. `cv.warpPerspective` ile ikinci görüntüyü birinciye hizala
6. İki görüntüyü bir canvas'ta birleştir
7. **Blend edge seam:** Örtüşen bölgede doğrusal ağırlık (alpha blend) uygula
8. 3+ görüntü için recursive stitching ekle

### Düşünmeniz Gereken Sorular:
- RANSAC algoritması neden 1981'de bilgisayar görüsünün en önemli buluşlarından biri sayılır?
- Homografi 3×3 matrisin 8 serbestlik derecesi neden 4 noktayla belirlenir?
- Multi-band blending nedir? Laplacian pyramid panorama stitching'i nasıl iyileştirir?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

def ozelikleri_cikart(img):
    orb = cv.ORB_create(nfeatures=5000)
    kp, desc = orb.detectAndCompute(img, None)
    return kp, desc

def eslesmeleri_bul(desc1, desc2):
    bf = cv.BFMatcher(cv.NORM_HAMMING)
    # Lowe ratio test
    pass

def panorama_olustur(img_list):
    # TODO
    pass
```

---
## ❓ Soru 3 — İnteraktif Grafik Editörü (Mini Paint)

**Zorluk: ⭐⭐⭐⭐☆**

OpenCV'nin çizim fonksiyonlarını kullanarak fare ile çalışan basit bir grafik editörü yazın. Undo/redo özelliği dahil olmalı.

### Görev:
1. Beyaz canvas oluştur (800×600)
2. Mouse callback ile çizim modlarını yönet:
   - `EVENT_LBUTTONDOWN` + drag → serbest çizim (line segmentler)
   - Araç seçimi: `r` (dikdörtgen), `c` (çember), `e` (elips), `p` (poligon nokta nokta), `t` (metin)
3. Renk paleti: ekranın altında 8 renk karesi — tıkla seç
4. Kalınlık ayarı: `1`-`9` tuşları
5. **Undo/Redo:** Her çizim işleminden önce `canvas.copy()` al, listeye ekle. `z` = undo, `y` = redo
6. `s` = PNG kaydet, `n` = yeni (temizle), `q` = çık
7. Dikdörtgen/elips çizerken anlık "rubber band" önizleme göster

### Düşünmeniz Gereken Sorular:
- Undo işlemi neden önceki durumu saklamayı gerektirir? Command pattern nedir?
- Anlık önizleme için her karede canvas'ı yeniden çizmek verimli mi? Daha iyi yolu var mı?
- Vektör grafikler (SVG) raster grafiklere göre hangi avantajlara sahip?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

canvas = np.ones((600, 800, 3), dtype=np.uint8) * 255
gecmis = [canvas.copy()]   # undo stack
gelecek = []               # redo stack
mod = "serbest"
renk = (0, 0, 0)
kalinlik = 3

def kaydet_durum():
    global gelecek
    gecmis.append(canvas.copy())
    gelecek = []

def undo():
    # TODO
    pass

def mouse_callback(event, x, y, flags, param):
    # TODO: mod'a göre çizim
    pass
```

---
## ❓ Soru 4 — Artırılmış Gerçeklik (AR): Belge Üzerine Sanal Etiket

**Zorluk: ⭐⭐⭐⭐⭐**

Kameradan gelen görüntüde belirli bir nesneyi (örn. dörtgen kağıt) tespit edip onun üstüne 3D gibi görünen sanal metin veya görüntü yerleştiren gerçek zamanlı AR uygulaması yazın.

### Görev:
1. Her karede `cv.findContours` ile en büyük dörtgen nesneyi tespit et
2. 4 köşesini `approxPolyDP` ile bul ve sırala
3. Bir sanal "etiket" görüntüsü hazırla (örn. logo, marka adı, 3D efekt ile kabartılmış yazı)
4. Etiketin 4 köşesini nesnenin 4 köşesine `getPerspectiveTransform` ile eşle
5. `warpPerspective` ile sanal etiketi dönüştür
6. Alpha maske ile sanal etiketi orijinal kareye blend et
7. Her karede tekrarla → nesne hareket ederken etiket onunla birlikte hareket etsin

### Düşünmeniz Gereken Sorular:
- ARKit ve ARCore bu temel üzerinde nasıl çok daha karmaşık sistemler kurdu?
- Kamera titremesini (jitter) azaltmak için dönüşüm matrisine nasıl temporal smoothing uygulanır?
- Marker-based AR (QR kod) ile markerless AR arasında hangi teknik farklar vardır?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

# Sanal etiket oluştur
etiket = np.zeros((200, 300, 3), dtype=np.uint8)
cv.putText(etiket, "OPENCV AR", (30, 120), cv.FONT_HERSHEY_SIMPLEX,
           1.5, (50, 200, 255), 3, cv.LINE_AA)

def dortgen_tespit(frame):
    """En büyük 4 köşeli nesneyi tespit et."""
    pass

video = cv.VideoCapture(0)
# TODO: döngü
```

---
## ❓ Soru 5 — Bullet-Time Efekti (360° Dondurulmuş Kamera Hareketi)

**Zorluk: ⭐⭐⭐⭐☆**

Matrix filmindeki "bullet-time" efektini görüntü işleme ile simüle edin. Tek bir kamera görüntüsünden, sanki sabit bir sahneyi farklı açılardan çekiliyormuş gibi bir video üretin.

### Görev:
1. Kameradan bir kare yakala (veya statik fotoğraf kullan)
2. Ana nesneyi (insan silueti) arka plandan ayır — basit yaklaşım: `cv.grabCut` veya renkli maskeleme
3. Sanal kamera yolu tanımla: merkez etrafında yay çizerek 0°'den 180°'ye
4. Her açı için:
   - Affine/perspektif dönüşüm uygula (eksenel dönüşü simüle et)
   - Arka planı ayrı bir kaydırma ile kaydır (paralaks efekti)
   - İki katmanı birleştir
5. Her kareyi video dosyasına yaz (`cv.VideoWriter`)
6. İsteğe bağlı: motion blur ekle (hareket hızıyla doğru orantılı)

### Düşünmeniz Gereken Sorular:
- Tek bir fotoğraftan 3D bilgi çıkarmak neden temel olarak mümkün değildir?
- Neural Radiance Fields (NeRF) bu problemi nasıl çözüyor?
- Matrix filmindeki orijinal bullet-time efekti nasıl çekildi? (120 kamera!)
- Paralaks (parallax) efekti nedir? Derinlik hissi nasıl yaratılır?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

def arka_plan_maskesi(frame):
    """İnsan silueti maskesi çıkar."""
    pass

def bullet_time_efekti(kare, aci):
    """Verilen açıya göre kamerayı döndürmüş gibi efekt uygula."""
    pass

kare = cv.imread("resim/portre.jpg")
fourcc = cv.VideoWriter_fourcc(*"XVID")
out = cv.VideoWriter("bullet_time.avi", fourcc, 30.0, (kare.shape[1], kare.shape[0]))

for aci in np.linspace(0, 180, 120):
    sonuc = bullet_time_efekti(kare, aci)
    out.write(sonuc)

out.release()
```

---
<div style="background: linear-gradient(135deg, #0a0510 0%, #1a0a2e 40%, #0f2d3a 100%);padding: 38px 42px; border-radius: 18px; text-align: center; margin-top: 20px; border: 1px solid #ff6b3544;"><h2 style="color: #ff6b35; font-family: 'Courier New', monospace; margin: 0 0 22px 0;">🎯 10. Hafta Özeti</h2><div style="color: #cdd6f4; text-align: left; max-width: 700px; margin: 0 auto; line-height: 2.2;">✅ <b>Affine dönüşüm:</b> 2×3 matris, 6 parametre — öteleme+döndürme+ölçekleme+shear<br>✅ <b>getRotationMatrix2D + warpAffine:</b> merkez etrafında açılı döndürme<br>✅ <b>Border modes:</b> CONSTANT, REPLICATE, REFLECT, REFLECT_101, WRAP<br>✅ <b>getAffineTransform:</b> 3 nokta çiftinden affine matrisi hesaplama<br>✅ <b>Perspektif dönüşüm:</b> 3×3 matris, 8 parametre — 4 nokta gerekli<br>✅ <b>getPerspectiveTransform + warpPerspective:</b> belge/fotoğraf düzleştirme<br>✅ <b>cv.line:</b> thickness, LINE_AA (Wu'nun anti-aliasing algoritması)<br>✅ <b>cv.rectangle:</b> nesne tespit sonuçları için bounding box<br>✅ <b>cv.circle, cv.ellipse:</b> tam/kısmi yay — Pacman animasyonu<br>✅ <b>cv.polylines:</b> çoklu çizgi/çokgen — np.int32 reshape(-1,1,2) formatı<br>✅ <b>cv.putText:</b> sadece ASCII — Türkçe için PIL gerekli<br>✅ <b>OpenCV + PIL hibrit:</b> TrueType fontlarla Unicode metin<br>✅ <b>Canlı kameraya dönüşüm:</b> gerçek zamanlı sürekli döndürme efekti</div><p style="color: #6b7280; margin: 22px 0 0 0; font-family: 'Courier New', monospace;">11. Hafta → Morfolojik İşlemler, Erosion, Dilation, Opening, Closing 🔧</p></div>